# Bottom Detection Method Comparison

Systematic comparison of all available bottom detection methods across
representative echogram files. Each method is run independently on the
same data, and the results are compared quantitatively (coverage,
smoothness, cross-method agreement) and visually (echogram overlays,
zoomed panels, agreement heatmaps).

In [ ]:
import sys
sys.path.insert(0, '../../..')
from project_paths import REPO_ROOT

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import load_zarr
from raw_sensor_integration.bottom_labelling import (
    detect_bottom_tvg,
    detect_bottom_dp,
    detect_bottom_gradient,
    detect_bottom_matched_filter,
    detect_bottom_temporal,
    detect_bottom_morphological,
    detect_bottom_cfar,
    detect_bottom_wavelet,
    detect_bottom_changepoint
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

ZARR_DIR = REPO_ROOT / 'Datasets/echopype_zarr'
FREQUENCY = 38000

## File Selection

A subset of zarr files chosen to span different recording conditions.
We load 10 files spread across the dataset to capture variation in
depth, bottom type, and noise conditions.

In [ ]:
all_zarrs = sorted(ZARR_DIR.glob('*.zarr'))
print(f'{len(all_zarrs)} zarr files available')

# Evenly spaced sample of 10 files
n_files = 10
indices = np.linspace(0, len(all_zarrs) - 1, n_files, dtype=int)
selected_zarrs = [all_zarrs[i] for i in indices]

for zp in selected_zarrs:
    print(f'  {zp.name}')

## Run All Detectors

In [ ]:
METHODS = {
    'tvg':          lambda p, r: detect_bottom_tvg(p, r, minimum_range=5.0),
    'dp':           lambda p, r: detect_bottom_dp(p, r, smoothness=1.0, minimum_range=5.0),
    'gradient':     lambda p, r: detect_bottom_gradient(p, r, minimum_range=5.0),
    'matched_filter': lambda p, r: detect_bottom_matched_filter(p, r, minimum_range=5.0),
    'temporal':     lambda p, r: detect_bottom_temporal(p, r, minimum_range=5.0),
    'morphological': lambda p, r: detect_bottom_morphological(p, r, minimum_range=5.0),
    'cfar':         lambda p, r: detect_bottom_cfar(p, r, minimum_range=5.0),
    'wavelet':      lambda p, r: detect_bottom_wavelet(p, r, minimum_range=5.0),
    'changepoint':  lambda p, r: detect_bottom_changepoint(p, r, minimum_range=5.0),
}

results = {}   # {file_name: {method: bottom_range_m}}
file_data = {} # {file_name: (power_db, range_array)}
timings = {}   # {method: [seconds_per_file]}

for method_name in METHODS:
    timings[method_name] = []

for zarr_path in selected_zarrs:
    fname = zarr_path.name
    print(f'\nProcessing {fname}...')

    data = load_zarr(zarr_path, frequency=FREQUENCY)
    power_db = data.sv_db
    range_array = data.range_array
    n_samples, n_pings = power_db.shape
    print(f'  Shape: ({n_samples}, {n_pings}), range: {range_array[0]:.1f}–{range_array[-1]:.1f} m')

    file_data[fname] = (power_db, range_array)
    results[fname] = {}

    for method_name, method_fn in METHODS.items():
        t0 = time.perf_counter()
        try:
            bottom = method_fn(power_db, range_array)
        except Exception as e:
            print(f'  {method_name}: FAILED — {e}')
            bottom = np.full(n_pings, np.nan)
        elapsed = time.perf_counter() - t0
        timings[method_name].append(elapsed)
        results[fname][method_name] = bottom

        valid_frac = np.sum(~np.isnan(bottom)) / n_pings
        print(f'  {method_name:16s}: {elapsed:6.2f}s  coverage={valid_frac:.1%}')

print('\nDone.')

## Per-File Echogram Overlays

Each file shown with all detection traces overlaid on the echogram,
colour-coded by method.

In [ ]:
METHOD_COLORS = {
    'tvg':           '#e41a1c',
    'dp':            '#377eb8',
    'gradient':      '#4daf4a',
    'matched_filter':'#984ea3',
    'temporal':      '#ff7f00',
    'morphological': '#a65628',
    'cfar':          '#f781bf',
    'wavelet':       '#ffff33',
    'changepoint':   '#00ced1',
}

for fname in results:
    power_db, range_array = file_data[fname]
    n_samples, n_pings = power_db.shape

    vmin, vmax = np.nanpercentile(power_db, [2, 98])

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(power_db, aspect='auto', cmap='viridis',
              vmin=vmin, vmax=vmax, origin='upper',
              extent=[0, n_pings, range_array[-1], range_array[0]])

    for method_name, bottom in results[fname].items():
        valid = ~np.isnan(bottom)
        if valid.sum() == 0:
            continue
        pings = np.arange(n_pings)
        ax.plot(pings[valid], bottom[valid],
                color=METHOD_COLORS[method_name],
                linewidth=1.0, alpha=0.8, label=method_name)

    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    ax.set_title(fname)
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()

## Zoomed Comparison at High-Disagreement Regions

For each file, find the 200-ping window where methods disagree most,
and show a zoomed view.

In [ ]:
ZOOM_WIDTH = 200

for fname in results:
    power_db, range_array = file_data[fname]
    n_samples, n_pings = power_db.shape

    # Stack all method traces
    all_traces = np.array([results[fname][m] for m in METHODS])
    # Per-ping std across methods (ignoring NaN)
    with np.errstate(all='ignore'):
        ping_std = np.nanstd(all_traces, axis=0)
    ping_std = np.nan_to_num(ping_std, nan=0.0)

    # Sliding window sum of std to find worst region
    if n_pings <= ZOOM_WIDTH:
        start = 0
    else:
        cumstd = np.cumsum(ping_std)
        window_sum = cumstd[ZOOM_WIDTH:] - cumstd[:-ZOOM_WIDTH]
        start = int(np.argmax(window_sum))
    end = min(start + ZOOM_WIDTH, n_pings)

    # Zoom depth around the median detection
    median_trace = np.nanmedian(all_traces, axis=0)
    valid_med = median_trace[start:end]
    valid_med = valid_med[~np.isnan(valid_med)]
    if len(valid_med) == 0:
        continue
    depth_center = np.median(valid_med)
    depth_margin = 20.0

    vmin, vmax = np.nanpercentile(power_db, [2, 98])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(power_db, aspect='auto', cmap='viridis',
              vmin=vmin, vmax=vmax, origin='upper',
              extent=[0, n_pings, range_array[-1], range_array[0]])

    for method_name, bottom in results[fname].items():
        valid = ~np.isnan(bottom)
        pings = np.arange(n_pings)
        ax.plot(pings[valid], bottom[valid],
                color=METHOD_COLORS[method_name],
                linewidth=1.2, alpha=0.85, label=method_name)

    ax.set_xlim(start, end)
    ax.set_ylim(depth_center + depth_margin, depth_center - depth_margin)
    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{fname} — zoomed (pings {start}–{end})')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
rows = []
for fname in results:
    power_db, range_array = file_data[fname]
    n_pings = power_db.shape[1]

    # Consensus: median across all methods
    all_traces = np.array([results[fname][m] for m in METHODS])
    consensus = np.nanmedian(all_traces, axis=0)

    for method_name in METHODS:
        bottom = results[fname][method_name]
        valid = ~np.isnan(bottom)
        coverage = valid.sum() / n_pings

        if valid.sum() < 2:
            rows.append({
                'file': fname, 'method': method_name,
                'coverage': coverage, 'median_depth_m': np.nan,
                'smoothness': np.nan, 'mad_vs_consensus': np.nan,
            })
            continue

        median_depth = np.nanmedian(bottom)
        # Smoothness: median absolute ping-to-ping change
        diffs = np.abs(np.diff(bottom[valid]))
        smoothness = np.median(diffs) if len(diffs) > 0 else np.nan

        # Deviation from consensus
        both_valid = valid & ~np.isnan(consensus)
        if both_valid.sum() > 0:
            mad = np.median(np.abs(bottom[both_valid] - consensus[both_valid]))
        else:
            mad = np.nan

        rows.append({
            'file': fname, 'method': method_name,
            'coverage': coverage, 'median_depth_m': median_depth,
            'smoothness': smoothness, 'mad_vs_consensus': mad,
        })

stats_df = pd.DataFrame(rows)

# Aggregate per method across files
summary = stats_df.groupby('method').agg({
    'coverage': 'mean',
    'smoothness': 'median',
    'mad_vs_consensus': 'median',
}).round(3)
summary.columns = ['mean_coverage', 'median_smoothness_m', 'median_mad_vs_consensus_m']
print(summary.sort_values('median_mad_vs_consensus_m').to_string())

## Cross-Method Agreement Heatmap

For each pair of methods, the median absolute difference in detected
bottom depth across all files and pings. Methods that cluster together
in the heatmap provide mutual validation.

In [ ]:
method_names = list(METHODS.keys())
n_methods = len(method_names)
agreement = np.zeros((n_methods, n_methods))

for fname in results:
    traces = {m: results[fname][m] for m in method_names}
    for i, m1 in enumerate(method_names):
        for j, m2 in enumerate(method_names):
            if i == j:
                continue
            both = ~np.isnan(traces[m1]) & ~np.isnan(traces[m2])
            if both.sum() > 0:
                diff = np.median(np.abs(traces[m1][both] - traces[m2][both]))
                agreement[i, j] += diff

agreement /= len(results)  # average over files

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(agreement, cmap='YlOrRd', origin='upper')
ax.set_xticks(range(n_methods))
ax.set_xticklabels(method_names, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n_methods))
ax.set_yticklabels(method_names, fontsize=8)

for i in range(n_methods):
    for j in range(n_methods):
        ax.text(j, i, f'{agreement[i,j]:.1f}',
                ha='center', va='center', fontsize=7)

plt.colorbar(im, ax=ax, label='Median abs. difference (m)')
ax.set_title('Cross-method agreement (lower = more similar)')
plt.tight_layout()
plt.show()

## Per-Ping Disagreement Analysis

For each ping across all files, count how many methods agree within
2 m of the median. Pings where fewer than half agree are the hard cases.

In [ ]:
AGREEMENT_TOL = 2.0  # meters

all_agreement_counts = []

for fname in results:
    all_traces = np.array([results[fname][m] for m in METHODS])
    consensus = np.nanmedian(all_traces, axis=0)
    n_pings = all_traces.shape[1]

    for p in range(n_pings):
        if np.isnan(consensus[p]):
            continue
        diffs = np.abs(all_traces[:, p] - consensus[p])
        n_agree = np.sum(diffs < AGREEMENT_TOL)
        all_agreement_counts.append(n_agree)

all_agreement_counts = np.array(all_agreement_counts)

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.arange(0, len(METHODS) + 2) - 0.5
ax.hist(all_agreement_counts, bins=bins, edgecolor='black', alpha=0.7)
ax.axvline(len(METHODS) / 2, color='red', linestyle='--', label='50% threshold')
ax.set_xlabel(f'Number of methods agreeing within {AGREEMENT_TOL} m')
ax.set_ylabel('Number of pings')
ax.set_title('Method agreement distribution across all files')
ax.legend()
plt.tight_layout()
plt.show()

hard_frac = np.mean(all_agreement_counts < len(METHODS) / 2)
print(f'{hard_frac:.1%} of pings have <50% method agreement')

## Computational Cost

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
method_names_sorted = sorted(METHODS.keys(), key=lambda m: np.mean(timings[m]))
mean_times = [np.mean(timings[m]) for m in method_names_sorted]
colors = [METHOD_COLORS[m] for m in method_names_sorted]

ax.barh(method_names_sorted, mean_times, color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Mean time per file (seconds)')
ax.set_title('Computational cost per method')

for i, (name, t) in enumerate(zip(method_names_sorted, mean_times)):
    ax.text(t + 0.05, i, f'{t:.2f}s', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## New Methods Contribution

Do CFAR, wavelet, and changepoint add value beyond the existing methods?
For each new method, compare its unique contribution to the ensemble:
cases where it agrees with consensus while some existing methods fail.

In [ ]:
NEW_METHODS = ['cfar', 'wavelet', 'changepoint']
OLD_METHODS = [m for m in METHODS if m not in NEW_METHODS]

for new_m in NEW_METHODS:
    n_unique_good = 0
    n_unique_bad = 0
    n_total_valid = 0

    for fname in results:
        all_traces = np.array([results[fname][m] for m in METHODS])
        consensus = np.nanmedian(all_traces, axis=0)
        new_trace = results[fname][new_m]
        n_pings = len(consensus)

        for p in range(n_pings):
            if np.isnan(consensus[p]) or np.isnan(new_trace[p]):
                continue
            n_total_valid += 1

            new_close = abs(new_trace[p] - consensus[p]) < AGREEMENT_TOL

            # How many old methods are close?
            old_vals = [results[fname][m][p] for m in OLD_METHODS]
            old_close = sum(1 for v in old_vals
                           if not np.isnan(v) and abs(v - consensus[p]) < AGREEMENT_TOL)
            old_total = sum(1 for v in old_vals if not np.isnan(v))

            if new_close and old_close < old_total * 0.5:
                n_unique_good += 1
            elif not new_close and old_close >= old_total * 0.5:
                n_unique_bad += 1

    print(f'{new_m}:')
    print(f'  Uniquely correct (agrees with consensus when >50% old methods fail): '
          f'{n_unique_good} pings ({n_unique_good/max(n_total_valid,1):.2%})')
    print(f'  Uniquely wrong (disagrees when >50% old methods agree): '
          f'{n_unique_bad} pings ({n_unique_bad/max(n_total_valid,1):.2%})')
    print()